# WAFER-GUARD · Fab-7 Excursion Detector — Pre-Release Model Audit

**Role:** you are the final technical reviewer on the release board.

**Status:** the model below clears the release gate (**F1 ≥ 0.90**). Deployment to the line-monitoring service is scheduled for Friday 06:00.

**Time-box:** 120 minutes.

## Your mandate

A staff engineer has a bad feeling about this notebook and has asked you to sign off — or to stop the release.

1. **Audit.** Identify *every* defect that invalidates the reported metric or would degrade the deployed model. For each defect give: (a) exact location, (b) mechanism, (c) **reproducible evidence from your own runs** — numbers, shapes, ablations. Assertions without evidence earn no credit. You are not told how many defects exist.
2. **Repair.** Deliver a corrected end-to-end pipeline that respects production causality: *no information from time *t′ > t* (or from the evaluation population) may influence a prediction made at time *t*.*
3. **Re-evaluate.** Report honest metrics at the **natural defect prevalence (~6.6 %)**. Justify your metric(s) and your decision-threshold procedure.
4. **Write-up.** ≤ 1 page: findings ranked by severity, residual risks, and your go / no-go recommendation.

> Deliverables: a repaired notebook + `AUDIT.md`. The existing notebook runs top-to-bottom without error. That is not evidence that it is correct.

### Data dictionary — `secom_telemetry_modified.csv`

| column | description |
|---|---|
| `sensor_0 … sensor_589` | 590 in-line process / metrology signals per wafer lot (anonymised) |
| `label` | electrical-test outcome: `-1` = pass, `+1` = fail |
| `timestamp` | lot start time (the fab runs continuously; equipment ages, recipes drift) |

**Fab tool-matching note (from the process-integration team):** a fraction of wafers are routed through *Probe Station PS-L (legacy)*, whose firmware emits telemetry in a different format than the current-generation stations. Consult the data.


## 0 · Environment

In [ ]:
!pip install "torch>=2.2" "scikit-learn>=1.2" "imbalanced-learn>=0.11" pandas numpy joblib matplotlib

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HP = dict(batch_size=64, epochs=40, lr=1e-3, weight_decay=1e-5,
          hidden=256, latent=32, lambda_clf=1.0, test_size=0.20)
print(f"torch {torch.__version__} | device={DEVICE}")

## 1. Data


In [ ]:
df = pd.read_csv("secom_telemetry_modified.csv")

SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
X_raw = df[SENSOR_COLS].to_numpy(dtype=np.float64)          # [N, 590]
y_raw = (df["label"].to_numpy() == 1).astype(np.int64)      # 1 = defective

print(f"X: {X_raw.shape} | defect rate: {y_raw.mean():.2%} "
      f"({y_raw.sum()} fail / {(1 - y_raw).sum()} pass)")
print(f"missing cells: {np.isnan(X_raw).mean():.2%}")

## 2 · Preprocessing & class rebalancing


In [ ]:
# --- missing-value imputation (fab telemetry has ~4 % sparse dropouts) ---
imputer = SimpleImputer(strategy="mean", keep_empty_features=True)
X_imp = imputer.fit_transform(X_raw)

# --- standardise: zero-mean / unit-variance per channel ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

# --- rebalance minority (defect) class ---
smote = SMOTE(sampling_strategy="auto", k_neighbors=5, random_state=SEED)
X_bal, y_bal = smote.fit_resample(X_scaled, y_raw)
print(f"after SMOTE: {X_bal.shape} | defect rate: {y_bal.mean():.2%}")

# --- hold-out split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=HP["test_size"], stratify=y_bal, random_state=SEED)
print(f"train {X_train.shape} | test {X_test.shape}")

## 3 · Tensors & loaders


In [ ]:
F = X_train.shape[1]                                     # 590

def make_loader(X, y, shuffle):
    # keep an explicit channel axis -> [N, 1, F]  (so a Conv1d encoder can be dropped in later)
    Xt = torch.from_numpy(X).float().unsqueeze(1)
    yt = torch.from_numpy(y).float()
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(TensorDataset(Xt, yt), batch_size=HP["batch_size"],
                      shuffle=shuffle, generator=g if shuffle else None, drop_last=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)
print(f"train batches: {len(train_loader)} | test batches: {len(test_loader)}")

## 4 · Model — supervised autoencoder


In [ ]:
class WaferSAE(nn.Module):
    def __init__(self, d_in: int, d_hid: int, d_lat: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_in),
        )
        self.head = nn.Linear(d_lat, 1)

    def forward(self, x):
        x = x.flatten(start_dim=1)                       # tolerate [B, F] or [B, C, F]
        z = self.encoder(x)
        return self.decoder(z), self.head(z).squeeze(-1), z

model = WaferSAE(F, HP["hidden"], HP["latent"]).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5 · Training


In [ ]:
opt   = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"])
bce   = nn.BCEWithLogitsLoss()

history = {"total": [], "recon": [], "clf": []}

for epoch in range(1, HP["epochs"] + 1):
    model.train()
    acc = dict(total=0.0, recon=0.0, clf=0.0); n_batches = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)

        reconstructed, logits, _ = model(batch_x)

        recon_loss = torch.mean((reconstructed - batch_x) ** 2)     # per-element MSE
        clf_loss   = bce(logits, batch_y)
        loss       = recon_loss + HP["lambda_clf"] * clf_loss

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        acc["total"] += loss.item(); acc["recon"] += recon_loss.item(); acc["clf"] += clf_loss.item()
        n_batches += 1

    sched.step()
    for k in acc:
        history[k].append(acc[k] / n_batches)
    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:02d}/{HP['epochs']} | total {history['total'][-1]:.4f} "
              f"| recon {history['recon'][-1]:.4f} | clf {history['clf'][-1]:.4f}")

## 6 · Evaluation


In [ ]:
model.eval()
probs = []
with torch.no_grad():
    for xb, _ in test_loader:
        _, logits, _ = model(xb.to(DEVICE))
        probs.append(torch.sigmoid(logits).cpu())
probs = torch.cat(probs).numpy()
preds = (probs >= 0.5).astype(int)

f1   = f1_score(y_test, preds)
prec = precision_score(y_test, preds)
rec  = recall_score(y_test, preds)
auc  = roc_auc_score(y_test, probs)
ap   = average_precision_score(y_test, probs)
tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()

print(f"F1 {f1:.4f} | precision {prec:.4f} | recall {rec:.4f} | ROC-AUC {auc:.4f} | AP {ap:.4f}")
print(f"confusion  TN={tn}  FP={fp}  FN={fn}  TP={tp}")



## 7 · Release gate



In [ ]:
GATE_F1 = 0.90
verdict = "PASSED ✅" if f1 >= GATE_F1 else "REVIEW ⚠️"
print(f"Release gate (F1 ≥ {GATE_F1}): {f1:.4f} → {verdict}")

os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/wafer_sae.pt")
joblib.dump({"imputer": imputer, "scaler": scaler}, "artifacts/preprocessor.joblib")
print("Artifacts written to ./artifacts/  — ready for the deployment pipeline.")